# StayNest - Session 6 Assignment (PySpark Deep Dive)
Work through the 8 tasks below in order. Read the Assignment Questions PDF for the
full detail and acceptance criteria. Fill in each `# TODO` cell, run it, and keep the
output visible. Run on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `customers.csv` to a Volume, then set `BASE`
to that path and run this cell. Counts should be 12000 / 200 / 2000.

In [0]:
# Point BASE at YOUR Volume path
BASE = "/Volumes/workspace/default/staynest"

print(spark.version)

read_csv = lambda name: (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df   = read_csv("bookings")
hotels_df     = read_csv("hotels")
customers_df  = read_csv("customers")

print(f"bookings: {bookings_df.count()}, "
      f"hotels: {hotels_df.count()}, "
      f"customers: {customers_df.count()}")

4.1.0
bookings: 12000, hotels: 200, customers: 2000


## Task 1 - Read and inspect
Show the schema, a few sample rows, the row count, and summary stats for the
numeric columns of `bookings_df`.

In [0]:
from pyspark.sql.functions import col

# Show schema
bookings_df.printSchema()

# Show sample rows
bookings_df.show(5)

# Row count (action → triggers Spark job)
print("Row count:", bookings_df.count())

# Summary stats for numeric columns
bookings_df.describe().show()


root
 |-- booking_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- hotel_id: integer (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- nights: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)

+----------+-----------+--------+------------+---------+------+--------+---------+
|booking_id|customer_id|hotel_id|booking_date|     city|nights|  amount|   status|
+----------+-----------+--------+------------+---------+------+--------+---------+
|   9000000|     701600|    3095|  2025-11-27|   Jaipur|     4| 6087.65|completed|
|   9000001|     700065|    3057|  2025-11-06|    Delhi|     1| 8211.19|cancelled|
|   9000002|     701392|    3187|  2025-08-21|   Jaipur|     2| 7176.52|cancelled|
|   9000003|     700867|    3112|  2025-03-22|Bengaluru|     5| 7880.62|completed|
|   9000004|     701521|    3043|  2025-04-19|   Mumbai|     5|21021.51|  pending|
+--------

## Task 2 - Select and filter
From `bookings_df`, select a few useful columns and return the **completed**
bookings with `amount` over 10000 in the cities Goa or Mumbai. Use `col()`, combine
conditions with `&`, and use `.isin(...)`.

In [0]:
from pyspark.sql.functions import col

filtered_df = (
    bookings_df
        .select(
            col("booking_id"),
            col("customer_id"),
            col("hotel_id"),
            col("city"),
            col("amount"),
            col("status")
        )
        .filter(
            (col("status") == "completed") &
            (col("amount") > 10000) &
            (col("city").isin("Goa", "Mumbai"))
        )
)

filtered_df.show()


+----------+-----------+--------+------+--------+---------+
|booking_id|customer_id|hotel_id|  city|  amount|   status|
+----------+-----------+--------+------+--------+---------+
|   9000007|     700868|    3127|Mumbai|15693.64|completed|
|   9000010|     700174|    3054|   Goa|30786.36|completed|
|   9000028|     701748|    3022|   Goa|27008.87|completed|
|   9000034|     700110|    3118|   Goa|64388.77|completed|
|   9000050|     701894|    3147|Mumbai|24275.14|completed|
|   9000061|     701454|    3115|Mumbai|19180.93|completed|
|   9000070|     701629|    3118|   Goa|58790.91|completed|
|   9000071|     700768|    3159|Mumbai|47548.35|completed|
|   9000075|     700312|    3006|   Goa|18043.52|completed|
|   9000087|     700953|    3099|   Goa|21529.88|completed|
|   9000090|     700655|    3125|   Goa|10372.32|completed|
|   9000094|     700142|    3184|   Goa|38954.16|completed|
|   9000096|     701426|    3099|   Goa|18394.79|completed|
|   9000110|     700819|    3148|Mumbai|

## Task 3 - Derived columns
Add: `amount_with_gst` (amount plus 12% tax), a `value_tier`
(premium / standard / budget) using `when`/`otherwise`, and a `booking_month`
from `booking_date`.

In [0]:
from pyspark.sql.functions import col, when, month

derived_df = (
    bookings_df
        .withColumn(
            "amount_with_gst",
            col("amount") * 1.12
        )
        .withColumn(
            "value_tier",
            when(col("amount") > 20000, "premium")
            .when(col("amount") > 10000, "standard")
            .otherwise("budget")
        )
        .withColumn(
            "booking_month",
            month(col("booking_date"))
        )
)

derived_df.show(5)


+----------+-----------+--------+------------+---------+------+--------+---------+-----------------+----------+-------------+
|booking_id|customer_id|hotel_id|booking_date|     city|nights|  amount|   status|  amount_with_gst|value_tier|booking_month|
+----------+-----------+--------+------------+---------+------+--------+---------+-----------------+----------+-------------+
|   9000000|     701600|    3095|  2025-11-27|   Jaipur|     4| 6087.65|completed|6818.168000000001|    budget|           11|
|   9000001|     700065|    3057|  2025-11-06|    Delhi|     1| 8211.19|cancelled|        9196.5328|    budget|           11|
|   9000002|     701392|    3187|  2025-08-21|   Jaipur|     2| 7176.52|cancelled|8037.702400000001|    budget|            8|
|   9000003|     700867|    3112|  2025-03-22|Bengaluru|     5| 7880.62|completed|        8826.2944|    budget|            3|
|   9000004|     701521|    3043|  2025-04-19|   Mumbai|     5|21021.51|  pending|       23544.0912|   premium|       

## Task 4 - Aggregations
For **completed** bookings, group by `city` and return: number of bookings, total
revenue, average amount, biggest booking, and the count of unique customers.
Order by revenue, highest first.

In [0]:
from pyspark.sql.functions import col, count, sum, avg, max, countDistinct

city_agg_df = (
    bookings_df
        .filter(col("status") == "completed")
        .groupBy(col("city"))
        .agg(
            count("*").alias("num_bookings"),
            sum(col("amount")).alias("total_revenue"),
            avg(col("amount")).alias("avg_amount"),
            max(col("amount")).alias("max_amount"),
            countDistinct(col("customer_id")).alias("unique_customers")
        )
        .orderBy(col("total_revenue").desc())
)

city_agg_df.show()


+---------+------------+--------------------+------------------+----------+----------------+
|     city|num_bookings|       total_revenue|        avg_amount|max_amount|unique_customers|
+---------+------------+--------------------+------------------+----------+----------------+
|      Goa|        2546| 4.459670178999999E7|17516.379336213664|  78481.24|            1441|
|   Mumbai|        1715|       3.624122112E7| 21131.90735860058|  78524.52|            1153|
|    Delhi|        1174| 2.631428154000001E7|22414.209148211252|  78669.69|             861|
|   Jaipur|         979|2.4436853129999984E7| 24961.03486210417|  76904.87|             796|
|Bengaluru|        1318| 2.267013697000002E7|  17200.4074127466|  78568.81|             969|
|  Udaipur|         691|1.2094427419999994E7| 17502.78931982633|   77939.5|             592|
|Rishikesh|         407|    8606121.57999999|21145.261867321846|  77458.26|             363|
|   Manali|         480|   6235480.680000003|12990.584750000007|  7729

## Task 5 - Joins
Inner-join bookings to hotels to enrich each booking. Do a left join too. Use
`left_anti` to check for orphaned bookings (expect 0). Then do a three-way join
with customers.

In [0]:
from pyspark.sql.functions import col

# --- 1. Inner join: bookings enriched with hotel info ---
inner_join_df = (
    bookings_df
        .join(
            hotels_df,
            on="hotel_id",
            how="inner"
        )
)

inner_join_df.show(5)


# --- 2. Left join: keep all bookings, enrich where possible ---
left_join_df = (
    bookings_df
        .join(
            hotels_df,
            on="hotel_id",
            how="left"
        )
)

left_join_df.show(5)


# --- 3. Left-anti join: orphaned bookings (expect 0) ---
orphans_df = (
    bookings_df
        .join(
            hotels_df,
            on="hotel_id",
            how="left_anti"
        )
)

orphans_df.show()   # should be empty


# --- 4. Three-way join: bookings + hotels + customers ---
three_way_df = (
    bookings_df
        .join(hotels_df, on="hotel_id", how="inner")
        .join(customers_df, on="customer_id", how="inner")
)

three_way_df.show(5)


+--------+----------+-----------+------------+---------+------+--------+---------+--------------+---------+--------+-----------+
|hotel_id|booking_id|customer_id|booking_date|     city|nights|  amount|   status|    hotel_name|     city|category|star_rating|
+--------+----------+-----------+------------+---------+------+--------+---------+--------------+---------+--------+-----------+
|    3095|   9000000|     701600|  2025-11-27|   Jaipur|     4| 6087.65|completed| Orchid Suites|   Jaipur|  Budget|        3.8|
|    3057|   9000001|     700065|  2025-11-06|    Delhi|     1| 8211.19|cancelled|Orchid Retreat|    Delhi|  Luxury|        4.1|
|    3187|   9000002|     701392|  2025-08-21|   Jaipur|     2| 7176.52|cancelled| Marigold Stay|   Jaipur| Premium|        3.8|
|    3112|   9000003|     700867|  2025-03-22|Bengaluru|     5| 7880.62|completed|    Grand Stay|Bengaluru|  Budget|        3.6|
|    3043|   9000004|     701521|  2025-04-19|   Mumbai|     5|21021.51|  pending|  Emerald Stay|

## Task 6 - Spark SQL + a window function
Register temp views and use `spark.sql` to get revenue by hotel `category` for
completed bookings. Then use a window function to rank the **top 3 hotels by
revenue within each city**.

In [0]:
# spark.sql(...) for revenue by category
bookings_df.createOrReplaceTempView("bookings")
hotels_df.createOrReplaceTempView("hotels")
customers_df.createOrReplaceTempView("customers")

category_revenue_df = spark.sql("""
    SELECT
        h.category,
        SUM(b.amount) AS total_revenue
    FROM bookings b
    INNER JOIN hotels h
        ON b.hotel_id = h.hotel_id
    WHERE b.status = 'completed'
    GROUP BY h.category
    ORDER BY total_revenue DESC
""")

category_revenue_df.show()


+--------+--------------------+
|category|       total_revenue|
+--------+--------------------+
|  Luxury|1.0683762689999975E8|
| Premium| 5.825564086000006E7|
|  Budget|2.2338253230000086E7|
+--------+--------------------+



In [0]:
# window function for top 3 hotels per city
top_hotels_df = spark.sql("""
    SELECT *
    FROM (
        SELECT
            h.city,
            h.hotel_id,
            h.hotel_name,
            SUM(b.amount) AS total_revenue,
            ROW_NUMBER() OVER (
                PARTITION BY h.city
                ORDER BY SUM(b.amount) DESC
            ) AS rank
        FROM bookings b
        INNER JOIN hotels h
            ON b.hotel_id = h.hotel_id
        WHERE b.status = 'completed'
        GROUP BY h.city, h.hotel_id, h.hotel_name
    )
    WHERE rank <= 3
    ORDER BY city, rank
""")

top_hotels_df.show()


+---------+--------+----------------+------------------+----+
|     city|hotel_id|      hotel_name|     total_revenue|rank|
+---------+--------+----------------+------------------+----+
|Anantapur|    3036|   Azure Retreat|1434328.3900000004|   1|
|Anantapur|    3189|   Lotus Retreat|         617708.27|   2|
|Anantapur|    3061|Orchid Residency|         205043.99|   3|
|Bengaluru|    3003|      Serene Inn|1633509.0900000005|   1|
|Bengaluru|    3162|   Willow Suites|1433492.0800000003|   2|
|Bengaluru|    3175|   Heritage Stay|        1400383.85|   3|
|    Delhi|    3057|  Orchid Retreat|2616612.5599999996|   1|
|    Delhi|    3104|   Marigold Stay|2327675.6300000004|   2|
|    Delhi|    3012|Orchid Residency|2264974.6499999994|   3|
|      Goa|    3118|    Palm Retreat| 3112412.420000001|   1|
|      Goa|    3033|       Lotus Inn|        3030434.53|   2|
|      Goa|    3184| Cedar Residency|2774119.2899999996|   3|
|   Jaipur|    3179|    Emerald Stay|        2686993.34|   1|
|   Jaip

## Task 7 - Write the result
Write your city-revenue result as **Parquet**, and also as a **Delta table** with
`saveAsTable`. Read the Delta table back to confirm.

In [0]:
city_agg_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/staynest/city_revenue_parquet"
)

city_agg_df.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/staynest/city_revenue_delta"
)

spark.read.format("delta").load(
    "/Volumes/workspace/default/staynest/city_revenue_delta"
).show()


+---------+------------+--------------------+------------------+----------+----------------+
|     city|num_bookings|       total_revenue|        avg_amount|max_amount|unique_customers|
+---------+------------+--------------------+------------------+----------+----------------+
|      Goa|        2546| 4.459670178999999E7|17516.379336213664|  78481.24|            1441|
|   Mumbai|        1715|       3.624122112E7| 21131.90735860058|  78524.52|            1153|
|    Delhi|        1174| 2.631428154000001E7|22414.209148211252|  78669.69|             861|
|   Jaipur|         979|2.4436853129999984E7| 24961.03486210417|  76904.87|             796|
|Bengaluru|        1318| 2.267013697000002E7|  17200.4074127466|  78568.81|             969|
|  Udaipur|         691|1.2094427419999994E7| 17502.78931982633|   77939.5|             592|
|Rishikesh|         407|    8606121.57999999|21145.261867321846|  77458.26|             363|
|   Manali|         480|   6235480.680000003|12990.584750000007|  7729

## Task 8 - One chained pipeline
In a single chain: keep completed bookings, join hotels, keep hotels with
`star_rating >= 4.0`, group by `city`, sum revenue, order descending, take the
top 5. End with one `.show()`.

In [0]:
from pyspark.sql.functions import col, sum

(
    bookings_df
        .filter(col("status") == "completed")
        .join(hotels_df.drop("city"), on="hotel_id", how="inner")
        .filter(col("star_rating") >= 4.0)
        .groupBy(col("city"))
        .agg(sum(col("amount")).alias("total_revenue"))
        .orderBy(col("total_revenue").desc())
        .limit(5)
        .show()
)


+---------+--------------------+
|     city|       total_revenue|
+---------+--------------------+
|      Goa|2.4725377899999995E7|
|   Mumbai|1.8937817669999998E7|
|    Delhi|1.8580418800000004E7|
|Bengaluru|    9129192.92999999|
|  Udaipur|          5916270.08|
+---------+--------------------+

